# Laboratory 2 — Building a Place Cell from Scratch

A reproducible mathematical experiment: **position → conditional rate → spike counts → occupancy-normalized spatial rate map**. We use synthetic trajectories; this does not demonstrate animal-like place-cell learning.

**Prerequisites:** NumPy, Matplotlib. **Duration:** ~30 minutes. Use `Run All` in Jupyter or Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2026)
ARENA_SIZE = 1.0   # meters
DT = 0.02          # seconds
DURATION = 300.0   # seconds
n_steps = int(DURATION / DT)
print(f"Simulating {n_steps:,} time bins ({DURATION:.0f} seconds).")

## 1. Gaussian field: a modeling assumption

For preferred position $\boldsymbol{\mu}$, baseline $r_{\min}$, peak $r_{\max}$, and width $\sigma$ (meters),

$$r(\mathbf{x}) = r_{\min} + (r_{\max}-r_{\min})\exp\bigl[-\|\mathbf{x}-\boldsymbol{\mu}\|^2/(2\sigma^2)\bigr].$$

In [ ]:
center = np.array([0.55, 0.48])  # meters
sigma = 0.14                       # meters
r_max = 25.0                       # Hz
r_min = 0.2                        # Hz

def place_rate(xy, center=center, sigma=sigma, r_max=r_max, r_min=r_min):
    xy = np.asarray(xy, dtype=float)
    dist2 = np.sum((xy - center)**2, axis=-1)
    return r_min + (r_max - r_min) * np.exp(-dist2/(2*sigma**2))

coord = np.linspace(0, ARENA_SIZE, 120)
X, Y = np.meshgrid(coord, coord)
rate_grid = place_rate(np.stack([X,Y], axis=-1))
fig, ax = plt.subplots(figsize=(5,4))
heat = ax.imshow(rate_grid, extent=[0,1,0,1], origin='lower', cmap='inferno')
ax.scatter(*center, marker='+', c='cyan', s=90, label='field center')
ax.set(xlabel='x (m)', ylabel='y (m)', title='Idealized place-cell expected firing rate')
ax.legend(loc='upper right'); fig.colorbar(heat, ax=ax, label='Expected rate (Hz)')
plt.show()

## 2. Generate a bounded, correlated exploration trajectory

Motion is a stochastic correlated walk with reflecting arena boundaries. Unlike independent random point samples, it produces a continuous trajectory. It is **not** a calibrated rodent or Jackal motion model.

In [ ]:
position = np.empty((n_steps, 2))
position[0] = [0.15, 0.15]
angle = 0.5
speed = 0.16  # m/s
for t in range(1, n_steps):
    angle += rng.normal(0, 0.18)  # heading diffusion per time step
    candidate = position[t-1] + speed*DT*np.array([np.cos(angle),np.sin(angle)])
    for d in range(2):
        if candidate[d] < 0:
            candidate[d] = -candidate[d]
            angle = np.pi-angle if d == 0 else -angle
        elif candidate[d] > ARENA_SIZE:
            candidate[d] = 2*ARENA_SIZE-candidate[d]
            angle = np.pi-angle if d == 0 else -angle
    position[t] = np.clip(candidate, 0, ARENA_SIZE)
fig, ax = plt.subplots(figsize=(5,5))
ax.plot(position[:,0], position[:,1], lw=0.5)
ax.set(xlim=(0,1),ylim=(0,1),xlabel='x (m)',ylabel='y (m)',title='Correlated random exploration')
ax.set_aspect('equal'); plt.show()

## 3. Convert rate to stochastic spike counts

In each time bin, $N[t]\sim\mathrm{Poisson}(r(\mathbf{x}_t)\Delta t)$. Multiple spikes per bin are allowed. We set a fixed random seed for reproducibility.

In [ ]:
expected_rate = place_rate(position)
spike_counts = rng.poisson(expected_rate * DT)
print('Total simulated spikes:', int(spike_counts.sum()))
print('Observed mean firing rate: %.2f Hz' % (spike_counts.sum()/DURATION))
print('Expected mean firing rate: %.2f Hz' % expected_rate.mean())
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(position[:,0],position[:,1],color='0.75',lw=0.45)
spike_pos = np.repeat(position, spike_counts, axis=0)
ax.scatter(spike_pos[:,0], spike_pos[:,1],c='darkred',s=3,alpha=0.45,label='spikes')
ax.set(xlim=(0,1),ylim=(0,1),xlabel='x (m)',ylabel='y (m)',title='Observed spikes along trajectory')
ax.set_aspect('equal'); ax.legend(); plt.show()

## 4. Occupancy-normalized rate map

A bin visited twice as long generally receives more spikes, even at identical firing rate. Estimate $\hat r_b=K_b/T_b$ with $K_b$ observed spike count and $T_b$ occupancy seconds. Unvisited bins are masked; we also mask bins with less than 0.2 s of occupancy.

In [ ]:
edges = np.linspace(0, ARENA_SIZE, 21)
occupancy_steps, _, _ = np.histogram2d(position[:,0],position[:,1],bins=[edges,edges])
spikes_per_bin, _, _ = np.histogram2d(position[:,0],position[:,1],bins=[edges,edges],weights=spike_counts)
occupancy_seconds = occupancy_steps * DT
min_occupancy = 0.2
rate_estimate = np.divide(spikes_per_bin,occupancy_seconds,
                          out=np.full_like(spikes_per_bin,np.nan),where=occupancy_seconds>=min_occupancy)
valid = np.isfinite(rate_estimate)
print('Coverage of adequately sampled spatial bins: %.1f%%' % (100*valid.mean()))
# Reference map evaluated at bin centers; sample rate-map errors only where the robot visited.
bin_centers = (edges[:-1]+edges[1:])/2
CX,CY = np.meshgrid(bin_centers,bin_centers,indexing='ij')
reference_rate = place_rate(np.stack([CX,CY],axis=-1))
if valid.any():
    mae = np.nanmean(np.abs(rate_estimate-reference_rate))
    print('MAE across adequately sampled bins: %.2f Hz' % mae)
fig,axes = plt.subplots(1,3,figsize=(13,3.8),constrained_layout=True)
for ax, field, title, cmap in zip(axes,
        [occupancy_seconds,reference_rate,rate_estimate],
        ['Occupancy (s)','Ground-truth rate (Hz)','Estimated rate (Hz)'],
        ['viridis','inferno','inferno']):
    im=ax.imshow(field.T,origin='lower',extent=[0,1,0,1],cmap=cmap)
    ax.set(title=title,xlabel='x (m)',ylabel='y (m)')
    fig.colorbar(im,ax=ax,shrink=.82)
plt.show()

## 5. Experiments to try

1. Change `sigma` from 0.14 to 0.25 m, rerun all cells, and compare field widths.
2. Change `DURATION` from 300 s to 60 s and examine spatial coverage and MAE.
3. Change `DT` from 0.02 s to 0.01 s, rerun, and compare **expected total spike counts** at constant duration.
4. Repeat with different seeds and report mean ± standard deviation for coverage and MAE.
5. **Conceptual:** Why do these firing maps not prove that the robot can infer its own position? Propose a population decoder and a held-out evaluation.

**Limitations:** No learning, remapping, refractory dynamics, or sensor uncertainty is modeled. Spatial position is supplied directly, not estimated from LiDAR. Subsequent labs will add these components.